<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# =============================================================================
# 50_build_app_artefacts.ipynb
# Purpose: consolidate baseline + keyword strategy outputs into artefacts the
#          Streamlit demo app expects to find and read without errors.
# Output : /content/drive/MyDrive/gt-markets/AppDemo/artefacts/<ASSET>/
# Notes  : - fixes "missing files" by guaranteeing metrics files exist per asset/freq
#          - fixes noisy/duplicated "strategy" names and mismatches between
#            signals_* and metrics_* files
#          - removes duplicates and normalizes casing in folder names
# =============================================================================

# --- 0) Setup: mount Drive and imports ----------------------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
import math
import re

# --- 1) Project paths ---------------------------------------------------------
PROJECT_DIR     = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT     = PROJECT_DIR / "app-demo" / "extracted"      # contains daily/weekly model outputs
DATA_PROCESSED  = PROJECT_DIR / "data" / "processed"          # merged market+trend data for baselines
DST_ARTE        = PROJECT_DIR / "AppDemo" / "artefacts"       # final artefacts for the app
DST_ARTE.mkdir(parents=True, exist_ok=True)

# --- 2) Configuration ---------------------------------------------------------
ASSETS      = ["GOLD", "BTC", "OIL", "USDCNY"]
FREQ_MAP    = {"daily": "D", "weekly": "W"}
ANNUAL      = {"D": 252, "W": 52}
COST_BPS    = 5  # cost per 100 bps; turnover cost = abs(diff(position)) * (COST_BPS/1e4)

# canonical asset normalizer (folder casing and inference)
ASSET_CANON = {"gold": "GOLD", "btc": "BTC", "oil": "OIL", "usdcny": "USDCNY"}

# columns the app expects to exist (even if empty) for each metrics file
METRIC_COLS = ["asset","freq","strategy","Return_Ann","Vol_Ann","Sharpe","MaxDD"]


# =============================================================================
# 3) Helpers: I/O, name sanitization, column detection, metrics
# =============================================================================

def _latest_processed_merged():
    """Pick the latest merged processed csv (main notebook output)."""
    files = sorted(DATA_PROCESSED.glob("merged_financial_trends_data_*.csv"))
    if not files:
        raise FileNotFoundError("No merged_financial_trends_data_*.csv found under data/processed")
    return files[-1]

def _canon_asset_from_string(s: str):
    s = s.lower()
    for k, v in ASSET_CANON.items():
        if k in s or f"{k}=" in s:
            return v
    return None

def _sanitize_strategy_name(text: str, asset: str, freq: str):
    """
    Build a short, stable strategy id for keyword runs.
    Strategy id must be consistent between signals_ and metrics_ rows.
    """
    # prefer parent directory as the "group" name if meaningful
    parts = [p for p in Path(text).parts if p not in ("/", "\\")]
    parent = parts[-2] if len(parts) >= 2 else ""
    stem   = Path(text).stem

    # candidate tokens: parent folder first (often the kw set), then filename stem
    raw = parent if len(parent) >= 3 and parent.lower() not in ("daily","weekly") else stem
    # normalize (letters/digits/underscore only), collapse repeats, upper-case
    raw = re.sub(r"[^A-Za-z0-9]+", "_", raw).strip("_").upper()
    # avoid overly long ids; keep the essence
    raw = raw[:18] if len(raw) > 18 else raw
    # final id: KW_<raw> (no asset/freq inside the id — those are separate columns)
    sid = f"KW_{raw}" if not raw.startswith("KW_") else raw
    # guard against empties
    if sid == "KW_":
        sid = "KW_MODEL"
    return sid

def _find_close_column(df: pd.DataFrame):
    """Find a price/close column with generous aliases (handles 'Adj Close')."""
    candidates = [c for c in df.columns]
    low = {c.lower(): c for c in candidates}
    # prefer common names
    for key in ("close", "price", "adj_close", "adjclose", "adj close", "last", "px_last"):
        if key in low:
            return low[key]
    # fallback: first numeric column that looks like a price
    for c in candidates:
        if pd.api.types.is_numeric_dtype(df[c]):
            return c
    return None

def _infer_position_series(df: pd.DataFrame):
    """
    Position inference:
      - 'position' (float/int) -> binarize to {0,1}
      - 'signal'   (-1/0/1)    -> convert to {0,1}
      - 'prob_up'  (0..1)      -> 1 if > 0.5 else 0
      - 'y_pred'/'pred' -> treat as 0/1 if already binary-like
    Returns Series of 0/1 floats. Empty Series if none found.
    """
    low = {c.lower(): c for c in df.columns}

    def _to_bin(s):
        s = pd.to_numeric(s, errors="coerce")
        # if looks like {-1,1} or {-1,0,1} => map -1 -> 0, 1 -> 1
        if s.dropna().isin([-1, 0, 1]).mean() > 0.9:
            s = s.replace({-1: 0}).clip(0, 1)
        # if probability => threshold
        if s.min() >= 0 and s.max() <= 1 and not s.dropna().isin([0,1]).all():
            s = (s > 0.5).astype(float)
        return s.fillna(0.0).clip(0, 1).astype(float)

    for key in ("position","signal","prob_up","up_prob","y_pred","pred"):
        if key in low:
            return _to_bin(df[low[key]])

    # nothing found
    return pd.Series(dtype=float)

def _equity_and_metrics(close: pd.Series, pos: pd.Series, freq: str):
    """
    Convert 0/1 position into equity curve and KPI metrics.
    Index alignment on Date is mandatory to avoid warnings.
    """
    close = pd.to_numeric(close, errors="coerce")
    ret   = close.pct_change(fill_method=None).fillna(0.0)

    pos   = pd.to_numeric(pos, errors="coerce").fillna(0.0).clip(0, 1)
    # explicit alignment on Date index
    pos   = pos.reindex(ret.index).fillna(0.0)

    turns = pos.diff().abs().fillna(0.0)
    cost  = turns * (COST_BPS / 1e4)

    strat = (pos.shift(1).fillna(0.0) * ret) - cost
    eq    = (1.0 + strat).cumprod()

    ann   = ANNUAL[freq]
    mu    = strat.mean() * ann
    sd    = strat.std(ddof=0) * math.sqrt(ann)
    sharpe= mu / sd if sd > 0 else np.nan
    maxdd = (eq / eq.cummax() - 1.0).min()

    return eq, {"Return_Ann": mu, "Vol_Ann": sd, "Sharpe": sharpe, "MaxDD": maxdd}

def _write_signals(out_dir: Path, strat: str, freq: str, date_idx: pd.Index, pos: pd.Series, close: pd.Series):
    """Emit signals_<STRAT>_<FREQ>.csv (step-style) and the equity figure."""
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "figs").mkdir(parents=True, exist_ok=True)

    # step signal: +1 => 1 (enter), -1 => 0 (exit)
    sig = pos.diff().fillna(0.0).replace({1.0: 1, -1.0: 0}).astype(int)

    out = pd.DataFrame({
        "Date": pd.to_datetime(date_idx),
        "signal": sig.values,
        "position": pos.astype(int).values,
        "Close": pd.to_numeric(close, errors="coerce").values,
        "strategy": str(strat)  # keep as text
    })
    out.to_csv(out_dir / f"signals_{strat}_{freq}.csv", index=False)

def _append_metrics_row(rows: list, asset: str, freq: str, strat: str, mets: dict):
    rows.append({
        "asset": asset,
        "freq": freq,
        "strategy": str(strat),
        "Return_Ann": mets["Return_Ann"],
        "Vol_Ann": mets["Vol_Ann"],
        "Sharpe": mets["Sharpe"],
        "MaxDD": mets["MaxDD"]
    })


# =============================================================================
# 4) Baselines (SMA/EMA) rebuilt fresh each run (fast; no ML)
# =============================================================================

def _load_merged_prices():
    """Load the merged processed file (main notebook output)."""
    df = pd.read_csv(_latest_processed_merged(), parse_dates=["Date"])
    df = df.set_index("Date").sort_index()
    return df

def _baseline_positions(close: pd.Series):
    """Two simple baselines: SMA(30>100) and EMA(12>26)."""
    out = {}
    sma30, sma100 = close.rolling(30).mean(), close.rolling(100).mean()
    out["BASE_SMA"] = (sma30 > sma100).astype(float)

    ema12, ema26 = close.ewm(span=12).mean(), close.ewm(span=26).mean()
    out["BASE_EMA"] = (ema12 > ema26).astype(float)
    return out

def build_baseline_artefacts():
    price_cols = {
        "GOLD": "GC=F Close",
        "BTC": "BTC-USD Close",
        "OIL": "CL=F Close",
        "USDCNY": "USDCNY=X Close"
    }
    df_prices = _load_merged_prices()

    for asset, col in price_cols.items():
        if col not in df_prices.columns:
            # create empty metrics files so the app doesn't complain
            (DST_ARTE / asset).mkdir(parents=True, exist_ok=True)
            for f in ("D","W"):
                pd.DataFrame(columns=METRIC_COLS).to_csv(DST_ARTE/asset/f"metrics_baseline_{f}.csv", index=False)
            continue

        close_full = df_prices[col].dropna()
        for freq in ("D","W"):
            # resampling for W if needed (use last price of the week)
            close = close_full if freq == "D" else close_full.resample("W-FRI").last().dropna()
            pos_dict = _baseline_positions(close)

            rows = []
            for strat, pos in pos_dict.items():
                # explicit Date index for alignment
                pos = pos.reindex(close.index).fillna(0.0)
                eq, mets = _equity_and_metrics(close, pos, freq)
                _append_metrics_row(rows, asset, freq, strat, mets)
                _write_signals(DST_ARTE/asset, strat, freq, close.index, pos, close)

                # quick equity figure
                ax = eq.plot(figsize=(6, 3), title=f"{asset} – {strat} – {freq}")
                ax.grid(True, alpha=0.3)
                ax.get_figure().savefig(DST_ARTE/asset/"figs"/f"{strat}_{freq}.png", dpi=140, bbox_inches="tight")
                plt.close(ax.get_figure())

            # overwrite (no duplicates)
            pd.DataFrame(rows, columns=METRIC_COLS).to_csv(DST_ARTE/asset/f"metrics_baseline_{freq}.csv", index=False)

    print("✅ Baseline artefacts built.")


# =============================================================================
# 5) Keyword (ML) artefacts from app-demo/extracted/daily|weekly
#     - enforce clean, short strategy ids: KW_<ID>
#     - align Date indexes to avoid noisy warnings
#     - deduplicate strategies per asset/freq
# =============================================================================

def _safe_read_csv(path: Path):
    try:
        df = pd.read_csv(path)
    except Exception:
        return pd.DataFrame()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
    return df

def build_keyword_artefacts():
    rows = []

    # scan both frequencies
    for subdir, freq in FREQ_MAP.items():
        src = SRC_EXTRACT / subdir
        if not src.exists():
            continue

        for csv in src.rglob("*.csv"):
            df = _safe_read_csv(csv)
            if df.empty or "Date" not in df.columns:
                continue

            asset = _canon_asset_from_string(csv.name) or _canon_asset_from_string(str(csv))
            if not asset:
                continue

            pos = _infer_position_series(df)
            if pos.empty:
                continue

            close_col = _find_close_column(df)
            if not close_col:
                continue

            # force Date index alignment to kill warnings
            close = pd.to_numeric(df[close_col], errors="coerce")
            close.index = pd.to_datetime(df["Date"])
            pos.index   = pd.to_datetime(df["Date"])
            close = close.dropna()
            pos   = pos.reindex(close.index).fillna(0.0)

            if close.empty:
                continue

            strat = _sanitize_strategy_name(str(csv), asset, freq)
            eq, mets = _equity_and_metrics(close, pos, freq)

            # write signals + fig
            _write_signals(DST_ARTE/asset, strat, freq, close.index, pos, close)
            ax = eq.plot(figsize=(6,3), title=f"{asset} – {strat} – {freq}")
            ax.grid(True, alpha=0.25)
            ax.get_figure().savefig(DST_ARTE/asset/"figs"/f"{strat}_{freq}.png", dpi=140, bbox_inches="tight")
            plt.close(ax.get_figure())

            # collect metrics row
            _append_metrics_row(rows, asset, freq, strat, mets)

    # write per-asset-per-freq metrics files; ensure dedup on (asset,freq,strategy)
    if rows:
        m = pd.DataFrame(rows, columns=METRIC_COLS).dropna(subset=["strategy"])
        # drop duplicated strategy ids within the same (asset,freq); keep last (latest scan)
        m = (m
             .sort_values(["asset","freq","strategy"])
             .drop_duplicates(["asset","freq","strategy"], keep="last"))
        for asset in ASSETS:
            a_dir = DST_ARTE / asset
            a_dir.mkdir(parents=True, exist_ok=True)
            for freq in ("D","W"):
                part = m.query("asset==@asset and freq==@freq")
                if part.empty:
                    # still emit an empty file with headers so the app won't complain
                    pd.DataFrame(columns=METRIC_COLS).to_csv(a_dir / f"metrics_keywords_{freq}.csv", index=False)
                else:
                    # enforce dtypes to avoid mixed-type 'strategy' column
                    part = part.copy()
                    part["strategy"] = part["strategy"].astype(str)
                    part.to_csv(a_dir / f"metrics_keywords_{freq}.csv", index=False)
    else:
        # no keyword runs found anywhere -> emit empty files for all
        for asset in ASSETS:
            (DST_ARTE / asset).mkdir(parents=True, exist_ok=True)
            for freq in ("D","W"):
                pd.DataFrame(columns=METRIC_COLS).to_csv(DST_ARTE/asset/f"metrics_keywords_{freq}.csv", index=False)

    print("✅ Keyword artefacts built.")


# =============================================================================
# 6) Folder normalization + duplicate cleanup
#     - merge mis-cased folders (Gold->GOLD, Oil->OIL)
#     - remove duplicated signals/figs by filename
# =============================================================================

def cleanup_and_normalize():
    # normalize asset folder casing
    for child in DST_ARTE.iterdir():
        if not child.is_dir():
            continue
        canon = ASSET_CANON.get(child.name.lower())
        if canon and child.name != canon:
            tgt = DST_ARTE / canon
            tgt.mkdir(parents=True, exist_ok=True)
            for f in child.rglob("*"):
                rel = f.relative_to(child)
                (tgt / rel.parent).mkdir(parents=True, exist_ok=True)
                if f.is_file():
                    shutil.move(str(f), str(tgt / rel))
            shutil.rmtree(child, ignore_errors=True)

    # dedup signals and figs by exact filename (keep latest)
    for asset in ASSETS:
        a_dir = DST_ARTE / asset
        if not a_dir.exists():
            continue

        # signals_*.csv
        seen = set()
        for f in sorted(a_dir.glob("signals_*_*.csv")):
            if f.name in seen:
                f.unlink(missing_ok=True)
            else:
                seen.add(f.name)

        # figs/*.png
        figs = a_dir / "figs"
        if figs.exists():
            seen = set()
            for f in sorted(figs.glob("*_*.png")):
                if f.name in seen:
                    f.unlink(missing_ok=True)
                else:
                    seen.add(f.name)

    print("✅ Normalization + dedup done.")


# =============================================================================
# 7) Validation table (quick ✓/✗ to spot gaps)
# =============================================================================

def validate_outputs():
    rows = []
    for asset in ASSETS:
        a_dir = DST_ARTE / asset
        for freq in ("D","W"):
            rows.append({
                "asset": asset,
                "freq": freq,
                "metrics_baseline": "✓" if (a_dir / f"metrics_baseline_{freq}.csv").exists() else "✗",
                "metrics_keywords": "✓" if (a_dir / f"metrics_keywords_{freq}.csv").exists() else "✗",
                "signals": "✓" if any(a_dir.glob(f"signals_*_{freq}.csv")) else "✗",
                "figs": "✓" if any((a_dir / "figs").glob(f"*_{freq}.png")) else "✗",
            })
    print(pd.DataFrame(rows).to_string(index=False))


# =============================================================================
# 8) Run the pipeline
# =============================================================================

build_baseline_artefacts()
build_keyword_artefacts()
cleanup_and_normalize()
validate_outputs()
print("✅ All artefacts ready at:", DST_ARTE)


Mounted at /content/drive
✅ Baseline artefacts built.
✅ Keyword artefacts built.
✅ Normalization + dedup done.
 asset freq metrics_baseline metrics_keywords signals figs
  GOLD    D                ✓                ✓       ✓    ✓
  GOLD    W                ✓                ✓       ✓    ✓
   BTC    D                ✓                ✓       ✓    ✓
   BTC    W                ✓                ✓       ✓    ✓
   OIL    D                ✓                ✓       ✓    ✓
   OIL    W                ✓                ✓       ✓    ✓
USDCNY    D                ✓                ✓       ✓    ✓
USDCNY    W                ✓                ✓       ✓    ✓
✅ All artefacts ready at: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
